# ML A/B Testing: LogReg vs RandomForest vs LightGBM on Dry Bean

This notebook walks through the experiment step by step. The heavy lifting (300+ model fits) is done
by `scripts/run_experiment.py`. This notebook loads the cached results, so it executes in seconds.
If no cache exists, it runs the experiment first (about 10 minutes).

**Design (pre-registered, see `docs/PLAN.md`):**
- Primary metric: macro-F1
- α = 0.05 with Holm correction
- Minimum practically meaningful difference (MPD) = 0.01

**Tests:**
- Corrected resampled t-test (primary)
- 5×2cv t / F
- McNemar
- Paired bootstrap
- TOST equivalence

In [ ]:
import sys, pickle, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from IPython.display import Image, display
from abtest import config
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
pd.set_option("display.width", 160)

cache = config.DATA_DIR / "results_cache.pkl"
if not cache.exists():
    from abtest.experiment import run, save
    from abtest.plots import make_all_figures
    res = run(); save(res); make_all_figures(res)
    pickle.dump(res, open(cache, "wb"))
res = pickle.load(open(cache, "rb"))
fig = lambda name: display(Image(filename=str(config.FIGURES_DIR / name), width=850))
print(json.dumps({k: v for k, v in res.meta.items() if k not in ("models", "versions")}, indent=1))

## 1. Data
The raw table has 13,611 beans with 16 shape features. Exact duplicates are dropped, then the data is split 80/20 into stratified development and hold-out sets.

In [ ]:
ds = res.dataset
print(f"raw rows={ds.n_raw:,}  duplicates dropped={ds.n_duplicates}  dev={len(ds.y_dev):,}  hold-out={len(ds.y_holdout):,}")
ds.X_dev.describe().T[["mean", "std", "min", "max"]]

In [ ]:
fig("01_class_distribution.png")

## 2. Cross-validated performance (10×10 repeated stratified CV)
The table shows the mean ± sd of each metric over 100 test folds. Every model saw **identical folds**, so the scores are paired.

In [ ]:
summary = res.cv.groupby("model")[["f1_macro","accuracy","precision_macro","recall_macro","f1_weighted","hamming_loss","mcc","cohen_kappa","log_loss","fit_seconds"]].agg(["mean","std"]).loc[config.MODEL_ORDER]
summary

In [ ]:
fig("02_cv_macro_f1_by_model.png")
fig("06_all_metrics_cv_mean.png")

**Hamming Loss check.** For single-label multi-class problems, Hamming Loss equals 1 − accuracy, as the next cell confirms:

In [ ]:
np.abs(res.cv["hamming_loss"] - (1 - res.cv["accuracy"])).max()

## 3. Primary test: corrected resampled t-test on macro-F1
The naive t-test is shown next to it to illustrate how badly it understates the uncertainty.

In [ ]:
cols = ["comparison","mean_diff","ci_low","ci_high","cohens_dz","wins_1","wins_2",
        "p_naive","p_corrected","p_holm_naive","p_holm_corrected"]
res.cv_tests[cols]

In [ ]:
fig("05_paired_fold_differences.png")
fig("03_forest_plot_differences.png")

## 4. Second opinion: 5×2cv paired t-test and combined F-test

In [ ]:
res.cv_tests[["comparison","t_5x2","p_5x2","p_holm_5x2","f_5x2_combined","p_5x2_combined_f","p_holm_5x2_combined_f"]]

In [ ]:
fig("04_pvalues_by_test.png")

## 5. Equivalence: TOST with ±0.01 margin
Equivalence is declared when the 90% CI lies entirely inside ±0.01.

In [ ]:
res.cv_tests[["comparison","mean_diff","tost_ci90_low","tost_ci90_high","tost_p","tost_equivalent"]]

## 6. Hold-out set: metrics, McNemar and paired bootstrap
Each model is fitted once on the full development set and scored once on the untouched hold-out set.

In [ ]:
res.holdout_metrics

In [ ]:
res.holdout_tests

In [ ]:
fig("10_mcnemar_discordant_pairs.png")
fig("09_bootstrap_holdout.png")

In [ ]:
fig("07_confusion_matrices_holdout.png")
fig("08_per_class_f1_holdout.png")

## 7. Secondary metrics (exploratory)
The corrected t-test is applied to every metric, with Holm correction within each metric. These results are exploratory: the decision rests on the primary metric only.

In [ ]:
sec = res.secondary_tests.pivot_table(index="metric_label", columns="comparison", values="mean_diff")
sig = res.secondary_tests.pivot_table(index="metric_label", columns="comparison", values="p_holm_within_metric")
print("Mean difference (model 1 − model 2):"); display(sec)
print("Holm-adjusted p (corrected t):"); display(sig)

## 8. Decision

In [ ]:
res.decisions

See the README for the written interpretation, limitations and the final recommendation.